In [3]:
import autogen
import os

# 配置 Gemini 3.1 Pro Preview (旗艦推理模型)
config_list = [{"model": "gemini-3.1-pro-preview", "api_key": os.environ.get("GEMINI_API_KEY"), "api_type": "google"}]
llm_config = {"config_list": config_list, "temperature": 0.7}

# ==============================================================================
# 1. 角色與身份分配表
# ==============================================================================
# 只有 Judge 知道這個全域資訊，Player 們只知道自己的身份
ROLE_MAP = {
    "Player_1": "狼人",
    "Player_2": "狼人",
    "Player_3": "預言家",
    "Player_4": "女巫",
    "Player_5": "平民",
    "Player_6": "平民"
}

alive_players = list(ROLE_MAP.keys())
agents = {}

# ==============================================================================
# 2. 初始化隱藏身份的 Players
# ==============================================================================
for player_name, role in ROLE_MAP.items():
    # 針對不同身份給予專屬的秘密指令
    if role == "狼人":
        secret_mission = "你是【狼人】。你的隊友是另一個狼人。晚上你們要殺人，白天你要假裝是平民或神職，把嫌疑推給別人。"
    elif role == "預言家":
        secret_mission = "你是【預言家】。晚上你可以查驗一人身份。白天你要利用查驗結果引導好人，但小心不要太早暴露被狼人殺死。"
    elif role == "女巫":
        secret_mission = "你是【女巫】。你有一瓶毒藥，晚上如果覺得某人是狼人，可以毒死他。白天請視情況帶領風向。"
    else:
        secret_mission = "你是【平民】。你沒有任何技能，只能在白天仔細聽大家發言，抓出邏輯漏洞並投票出狼人。"

    agents[player_name] = autogen.AssistantAgent(
        name=player_name,
        system_message=f"你的名字是 {player_name}。{secret_mission}\n請務必用第一人稱發言，且絕對不能在不必要時暴露真實身份。",
        llm_config=llm_config
    )

# 遊戲主持人 (上帝)
judge = autogen.UserProxyAgent(
    name="Judge",
    system_message="遊戲主持人，負責掌控流程與判定勝負。",
    human_input_mode="NEVER",
    code_execution_config=False
)

# ==============================================================================
# 3. 遊戲主迴圈 (包含勝負判定)
# ==============================================================================
def check_win_condition():
    wolves_alive = sum(1 for p in alive_players if ROLE_MAP[p] == "狼人")
    good_alive = len(alive_players) - wolves_alive
    
    if wolves_alive == 0:
        print("\n🎉 遊戲結束：所有狼人皆已死亡，【好人陣營勝利】！")
        return True
    elif wolves_alive >= good_alive:
        print("\n💀 遊戲結束：狼人數量大於等於好人，【狼人陣營勝利】！")
        return True
    return False

def run_game(days=10):
    day_count = 1
    
    while day_count <= days:
        print(f"\n{'='*40}\n🌙 第 {day_count} 天夜晚：天黑請閉眼\n{'='*40}")
        night_deaths = []

        # [狼人階段]
        # 為了簡化 Demo，我們讓 Player_1 (狼人代表) 決定殺誰
        if "Player_1" in alive_players:
            kill_chat = judge.initiate_chat(agents["Player_1"], message=f"現在是晚上。目前存活玩家：{alive_players}。請直接回覆一個你要殺害的玩家名字 (例如 Player_5)。", max_turns=1)
            target = kill_chat.chat_history[-1]['content'].strip()
            night_deaths.append(target)
            print("🐺 狼人已經行動...")

        # [預言家階段]
        if "Player_3" in alive_players:
            seer_chat = judge.initiate_chat(agents["Player_3"], message=f"預言家請睜眼。目前存活：{alive_players}。請回覆你要查驗誰？", max_turns=1)
            check_target = seer_chat.chat_history[-1]['content'].strip()
            # 模擬上帝告知結果
            is_wolf = "是" if ROLE_MAP.get(check_target) == "狼人" else "不是"
            judge.send(f"查驗結果：{check_target} {is_wolf} 狼人。", agents["Player_3"])
            print("👁️ 預言家已經行動...")

        # [女巫階段] (簡化版：只使用毒藥)
        if "Player_4" in alive_players:
            witch_chat = judge.initiate_chat(agents["Player_4"], message=f"女巫請睜眼。你要使用毒藥嗎？如果要，請回覆玩家名字；如果不要，請回覆『放棄』。", max_turns=1)
            poison_target = witch_chat.chat_history[-1]['content'].strip()
            if "Player" in poison_target:
                night_deaths.append(poison_target)
            print("🧪 女巫已經行動...")

        # 處理死亡名單
        for dead in set(night_deaths):
            if dead in alive_players:
                alive_players.remove(dead)

        if check_win_condition(): break

        print(f"\n{'='*40}\n☀️ 第 {day_count} 天白天：天亮請睜眼\n{'='*40}")
        if night_deaths:
            print(f"📢 昨晚死亡的玩家是：{', '.join(night_deaths)}")
        else:
            print("📢 昨晚是平安夜！沒有人死亡。")

        # [白天討論階段] - 只讓活著的玩家進入群聊
        alive_agents = [judge] + [agents[p] for p in alive_players]
        
        groupchat = autogen.GroupChat(
            agents=alive_agents,
            messages=[],
            max_round=days*(len(alive_players))+1, # 每人發言一次 + 總結
            speaker_selection_method="auto"
        )
        manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)
        
        print("\n💬 開始白天自由辯論...")
        judge.initiate_chat(manager, message="請大家開始根據昨晚的情況進行推理辯論！每人都要發言，揪出狼人是誰。")

        # [投票階段] (為了腳本流暢度，這裡由主持人直接隨機/邏輯模擬踢出一人，實務上可再開一個迴圈讓大家投票)
        print("\n🗳️ 辯論結束，進入投票環節！")
        # 假設 AI 透過辯論決定投出 alive_players 中的某一人
        # (這裡做一個簡化模擬，實際工作坊可以讓 Judge 問大家要投誰並做字串統計)
        voted_out = alive_players[-1] # 模擬某人被高票投出
        print(f"🔨 投票結果：{voted_out} 被處決！")
        alive_players.remove(voted_out)

        if check_win_condition(): break
        
        day_count += 1

if __name__ == "__main__":
    run_game()


🌙 第 1 天夜晚：天黑請閉眼
Judge (to Player_1):

現在是晚上。目前存活玩家：['Player_1', 'Player_2', 'Player_3', 'Player_4', 'Player_5', 'Player_6']。請直接回覆一個你要殺害的玩家名字 (例如 Player_5)。

--------------------------------------------------------------------------------
Player_1 (to Judge):

Player_3

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (2c49390e-ef76-4e50-96fc-87f034d332f2): Maximum turns (1) reached
🐺 狼人已經行動...
Judge (to Player_3):

預言家請睜眼。目前存活：['Player_1', 'Player_2', 'Player_3', 'Player_4', 'Player_5', 'Player_6']。請回覆你要查驗誰？

--------------------------------------------------------------------------------


/Users/max/Desktop/python_codes/autogen-ai-company-framework/autogen-env/lib/python3.11/site-packages/autogen/oai/gemini.py:1106: UserWarning: Cost calculation is not implemented for model gemini-3.1-pro-preview. Cost will be calculated zero.
  warnings.warn(


Player_3 (to Judge):

我要查驗 Player_2。

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (80f0bcf5-5977-4406-8edc-da7ce1666d26): Maximum turns (1) reached
Judge (to Player_3):

查驗結果：我要查驗 Player_2。 不是 狼人。

--------------------------------------------------------------------------------
👁️ 預言家已經行動...
Judge (to Player_4):

女巫請睜眼。你要使用毒藥嗎？如果要，請回覆玩家名字；如果不要，請回覆『放棄』。

--------------------------------------------------------------------------------
Player_4 (to Judge):

放棄

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (430f7e87-9cc0-4aa3-b16f-15d085f9294f): Maximum turns (1) reached
🧪 女巫已經行動...

☀️ 第 1 天白天：天亮請睜眼
📢 昨晚死亡的玩家是：Player_3

💬 開始白天自由辯論...
Judge (to chat_manager):

請大家開始根據昨晚的情況進行推理辯論！每人都要發言，揪出狼人是誰。

--------------------------------------------------------------------------------

Next speaker: Player_1

Player_1 (to chat_manager):

大家早安，我是 Player_1。

既然天亮了，我們就得好好盤一下邏輯。我先表